# What CAN'T Be Done on Snowpark Connect — Snowpark Connect Edition

This notebook runs the **same code** as `04_unsupported_classic.ipynb` but on **Snowpark Connect**.

Each cell is wrapped in `try/except` so the notebook runs end-to-end — you see the **actual error messages**.

| # | Feature | Classic Spark | Snowpark Connect | Snowflake Alternative |
|---|---------|:---:|:---:|---|
| 1 | RDDs | Works | **FAILS** | Use DataFrame API |
| 2 | Structured Streaming | Works | **FAILS** | Snowpipe Streaming / Dynamic Tables |
| 3 | MLlib | Works | **FAILS** | Snowpark ML / Cortex ML |
| 4 | SparkContext (broadcast, accumulator) | Works | **FAILS** | Not needed — use DataFrames |
| 5 | Repartition / Coalesce | Works | No-op (silently ignored) | Snowflake auto-optimizes |

## Session Setup — Snowpark Connect

In [1]:
from snowflake import snowpark_connect

spark = snowpark_connect.init_spark_session()

# Quick test — this WORKS
df = spark.createDataFrame([(1, "test")], ["id", "msg"])
df.show()
print("Session ready. DataFrame API works fine.")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/snowflake/connector/vendored/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/vyadav/Library/Python/3.11/lib/python/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
2026-03-10 21:23:25,938 - snowflake_connect_server - WARNING - [Thread 8499998848] - OpenTelemetry basic modules not available: No module named 'opentelemetry'
2026-03-10 21:23:25,940 - snowflake_connect_server - INFO - [Thread 8499998848] - JAVA_HOME=/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
2026-03-10 21:23:29,211 - snowflake_connect_server - INFO - [Thread 14143221760] - Configuring session <snowflake.snowpark.session.Session: account="SFSEAPAC-VYADAV_AWS_AU

+---+----+
| id| msg|
+---+----+
|  1|test|
+---+----+

Session ready. DataFrame API works fine.


---
## 1. RDDs — FAILS on Snowpark Connect
RDDs require `SparkContext`, which does not exist in Snowpark Connect. Only the DataFrame API is supported.

In [2]:
try:
    sc = spark.sparkContext
    rdd = sc.parallelize([1, 2, 3, 4, 5])
    print(rdd.collect())
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")
    print()
    print("WHY: SparkContext is not available in Snowpark Connect.")
    print("     RDDs (parallelize, map, reduce, filter) cannot be used.")
    print("FIX: Rewrite RDD code using the DataFrame API.")

FAILED: PySparkNotImplementedError: [NOT_IMPLEMENTED] sparkContext() is not implemented.

WHY: SparkContext is not available in Snowpark Connect.
     RDDs (parallelize, map, reduce, filter) cannot be used.
FIX: Rewrite RDD code using the DataFrame API.


---
## 2. Structured Streaming — FAILS on Snowpark Connect
`readStream` / `writeStream` are not supported. Snowflake handles streaming via Snowpipe Streaming and Dynamic Tables.

In [3]:
try:
    streaming_df = (
        spark.readStream
        .format("rate")
        .option("rowsPerSecond", 5)
        .load()
    )
    print(f"Is streaming: {streaming_df.isStreaming}")
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")
    print()
    print("WHY: Structured Streaming (readStream/writeStream) is not supported.")
    print("FIX: Use Snowpipe Streaming for real-time ingestion.")
    print("     Use Dynamic Tables for incremental materialization.")
    print("     Use OpenFlow for streaming connectors.")

Is streaming: False


---
## 3. MLlib — FAILS on Snowpark Connect
`pyspark.ml` is not supported. Snowflake provides Snowpark ML and Cortex ML as alternatives.

In [4]:
try:
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml import Pipeline

    training_data = spark.createDataFrame([
        (0, 1.0, 0.1, 0),
        (1, 0.2, 0.8, 1),
        (2, 0.9, 0.2, 0),
        (3, 0.1, 0.9, 1),
        (4, 0.8, 0.3, 0),
        (5, 0.3, 0.7, 1),
    ], ["id", "feature1", "feature2", "label"])

    assembler = VectorAssembler(inputCols=["feature1", "feature2"], outputCol="features")
    lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
    pipeline = Pipeline(stages=[assembler, lr])

    model = pipeline.fit(training_data)
    predictions = model.transform(training_data)
    predictions.select("id", "label", "prediction").show()
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")
    print()
    print("WHY: pyspark.ml (MLlib) is not supported on Snowpark Connect.")
    print("FIX: Use Snowpark ML for model training and deployment.")
    print("     Use Snowflake Model Registry for model management.")
    print("     Use Cortex ML Functions for built-in models (forecasting, anomaly detection).")

FAILED: AssertionError: 

WHY: pyspark.ml (MLlib) is not supported on Snowpark Connect.
FIX: Use Snowpark ML for model training and deployment.
     Use Snowflake Model Registry for model management.
     Use Cortex ML Functions for built-in models (forecasting, anomaly detection).


---
## 4. SparkContext Direct Access — FAILS on Snowpark Connect
Broadcast variables, accumulators, and `textFile` all require `SparkContext` which is not available.

In [5]:
# --- Broadcast ---
try:
    sc = spark.sparkContext
    lookup = {"A": "Apple", "B": "Banana"}
    broadcast_lookup = sc.broadcast(lookup)
    print(broadcast_lookup.value)
except Exception as e:
    print(f"BROADCAST FAILED: {type(e).__name__}: {e}")

print()

# --- Accumulator ---
try:
    sc = spark.sparkContext
    counter = sc.accumulator(0)
    print(counter.value)
except Exception as e:
    print(f"ACCUMULATOR FAILED: {type(e).__name__}: {e}")

print()

# --- textFile ---
try:
    sc = spark.sparkContext
    lines = sc.textFile("/tmp/test.txt")
    print(lines.count())
except Exception as e:
    print(f"TEXTFILE FAILED: {type(e).__name__}: {e}")

print()
print("WHY: SparkContext does not exist in Snowpark Connect.")
print("FIX: Use DataFrame API for all operations.")
print("     Broadcast joins are handled automatically by Snowflake's optimizer.")

BROADCAST FAILED: PySparkNotImplementedError: [NOT_IMPLEMENTED] sparkContext() is not implemented.

ACCUMULATOR FAILED: PySparkNotImplementedError: [NOT_IMPLEMENTED] sparkContext() is not implemented.

TEXTFILE FAILED: PySparkNotImplementedError: [NOT_IMPLEMENTED] sparkContext() is not implemented.

WHY: SparkContext does not exist in Snowpark Connect.
FIX: Use DataFrame API for all operations.
     Broadcast joins are handled automatically by Snowflake's optimizer.


---
## 5. Repartition / Coalesce — No-op on Snowpark Connect
These calls don't error — they are **silently ignored**. Snowflake handles data distribution automatically.

In [6]:
df = spark.createDataFrame([(i, f"row_{i}") for i in range(100)], ["id", "value"])

# These won't error, but they also won't do anything
df_8 = df.repartition(8)
df_2 = df_8.coalesce(2)

# The data is still there — the calls were just ignored
print(f"Row count after repartition + coalesce: {df_2.count()}")
print()
print("NOTE: repartition() and coalesce() are silently ignored.")
print("      Snowflake manages data distribution automatically.")
print("      No action needed — just remove these calls if you want clean code.")

Row count after repartition + coalesce: 100

NOTE: repartition() and coalesce() are silently ignored.
      Snowflake manages data distribution automatically.
      No action needed — just remove these calls if you want clean code.


---
## Summary

| Feature | Error on Snowpark Connect | Snowflake Alternative |
|---|---|---|
| **RDDs** (parallelize, map, reduce) | `SparkContext` not available | DataFrame API (fully supported) |
| **Structured Streaming** (readStream) | `readStream` not supported | Snowpipe Streaming / Dynamic Tables |
| **MLlib** (Pipeline, models) | `pyspark.ml` not supported | Snowpark ML / Cortex ML |
| **SparkContext** (broadcast, accumulator) | `SparkContext` not available | Snowflake optimizer handles broadcasts |
| **Repartition / Coalesce** | No error — silently ignored | Snowflake auto-optimizes partitioning |

### Key Takeaway

> If your code uses **DataFrames + SQL + Python UDFs**, it works on Snowpark Connect with **zero changes**.
>
> If your code uses **RDDs, Streaming, or MLlib**, those parts need to be replaced with Snowflake-native equivalents.

In [7]:
spark.stop()
print("Session stopped.")

Session stopped.
